In [1]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.optuna_objective import create_objective
from src.utils.telegram import send_message

In [2]:
# === Configuration === (you edit here) ===
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "cb"
    data_id: str = "057"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 20
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Initial params
    use_initial: Literal["never", "manual"] = "never"
    initial_param_sources: list[tuple[str, int]] = field(default_factory=list)   # (study_name, n_trial) 例: ("xgb-001", 1)

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

    # Option
    opts: dict = field(default_factory=dict)


cfg = Config()
cfg.initial_param_sources = [("xgb-057", 5)]

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# === Build & Run (frozen) ===
# --- helper: initial params loader ---
def load_initial_params(sources: list[tuple[str, int]]) -> list[dict]:
    loaded = []
    for study, n_trial in sources:
        path = Path(f"../../artifacts/optuna/{study}/trl{n_trial}.json")
        with path.open("r") as f:
            params = json.load(f)["params"]
        loaded.append(params)
    return loaded


# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


objective = create_objective(
    cfg.model_name,
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=cfg.opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

initial_params = None
if cfg.use_initial == "manual":
    initial_params = load_initial_params(cfg.initial_param_sources)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params=initial_params
)

[I 2025-10-06 10:47:11,595] Using an existing study with name 'lgbm-057' instead of creating a new one.


[initial] none


  0%|          | 0/20 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 15.28 GB
Free GPU Mem: 6.81 GB
Training until validation scores don't improve for 500 rounds
[100]	train's auc: 0.967336	valid's auc: 0.968041
[200]	train's auc: 0.970251	valid's auc: 0.970737
[300]	train's auc: 0.972098	valid's auc: 0.97241
[400]	train's auc: 0.973487	valid's auc: 0.973614
[500]	train's auc: 0.974394	valid's auc: 0.974315
[600]	train's auc: 0.975078	valid's auc: 0.974765
[700]	train's auc: 0.97562	valid's auc: 0.97508
[800]	train's auc: 0.976097	valid's auc: 0.975328
[900]	train's auc: 0.976509	valid's auc: 0.975524
[1000]	train's auc: 0.976896	valid's auc: 0.975685
[1100]	train's auc: 0.977251	valid's auc: 0.975814
[1200]	train's auc: 0.977569	valid's auc: 0.975921
[1300]	train's auc: 0.977877	valid's auc: 0.976025
[1400]	train's auc: 0.978156	valid's auc: 0.976112
[1500]	train's auc: 0.978421	valid's auc: 0.976185
[1600]	train's auc: 0.97868	valid's auc: 0.976237
[1700]	train's auc: 0.978928	valid's auc: 0.976296
[1800]	train's auc:

iter_f1,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇██
train/f1/auc,▁▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
valid/f1/auc,▁▄▄▅▆▆▇▇▇███████████████████████████████
auc_f1,0.97683
iter_f1,5602
runtime_f1,9.41483
train/f1/auc,0.98559
valid/f1/auc,0.97682


[I 2025-10-06 10:57:27,812] Trial 1 finished with value: 0.9768330962308425 and parameters: {'learning_rate': 0.02, 'num_leaves': 637, 'min_child_samples': 19020, 'min_split_gain': 0.24658329458549094, 'feature_fraction': 0.4197316968394073, 'bagging_fraction': 0.8234027960663655, 'bagging_freq': 3, 'lambda_l1': 2.231010801867923e-05, 'lambda_l2': 1.574189004745663}. Best is trial 1 with value: 0.9768330962308425.


Fold Col: 5fold-s42
Free CPU Mem: 7.29 GB
Free GPU Mem: 6.87 GB
Training until validation scores don't improve for 500 rounds
[100]	train's auc: 0.968305	valid's auc: 0.968874
[200]	train's auc: 0.971157	valid's auc: 0.971488
